# Credit Risk Scorecard — SQL Data Extraction Simulation

**Author:** Mathias Alejandro Gómez Chan
**Date:** April 2026
**GitHub:** [Mathias70473](https://github.com/Mathias70473)

---

> **Note:** This notebook is numbered `00` because it has a direct impact in the later Exploratory Data Analysis. The goal is to replicate a production environment, 
> data extraction happens before any analysis. It feeds directly into 
> `01-eda.ipynb`.


In [78]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('../data/raw/german_credit_data.csv')

# Create SQLite database in memory
conn = sqlite3.connect(':memory:')

# Write dataframe to SQL table
df.to_sql('german_credit', conn, index=False, if_exists='replace')

print("Database created successfully")
print(f"Table shape: {df.shape}")

Database created successfully
Table shape: (1000, 21)


## Production Context

In a production credit risk environment, loan data does not arrive as a flat CSV. 
It lives in relational databases — typically joining tables for loan origination, 
customer demographics, credit bureau data, and transaction history.

This notebook simulates that extraction layer using SQLite. The German Credit 
Dataset is loaded into an in-memory SQL database, queried with analytical 
credit risk questions, and the clean output is passed to the EDA notebook.

**This approach reflects how production pipelines actually work:**


## Query 1 — Default Rate Overview

**Business question:** What is the overall distribution of good and bad 
customers in the portfolio?

**Why it matters:** Before any modeling, a credit risk team needs to understand 
the baseline default rate. A 30% default rate is significantly higher than 
a typical retail portfolio (usually 5–15%), which informs decisions about 
model calibration and approval thresholds.

In [79]:
# Query 1 — Default rate overview
query1 = """
SELECT 
    kredit,
    COUNT(*) as total
FROM german_credit
GROUP BY kredit
"""

result1 = pd.read_sql(query1, conn)
result1['label'] = result1['kredit'].map({0: 'Bad (default)', 1: 'Good'})
print("Query 1 — Default rate overview")
print(result1)

Query 1 — Default rate overview
   kredit  total          label
0       0    300  Bad (default)
1       1    700           Good


## Query 2 — Default Rate by Checking Account Status

**Business question:** How does default rate vary across checking account 
status categories?

**Why it matters:** Checking account status is the strongest predictor in 
this dataset (IV = 0.666). This query previews that signal directly in SQL. A credit risk team would use this to validate 
that the variable behaves as expected in the raw data.

In [80]:
# Query 2 — Default rate by checking account status
query2 = """
SELECT 
    laufkont as checking_status,
    kredit as default_flag,
    COUNT(*) as total
FROM german_credit
GROUP BY laufkont, kredit
ORDER BY laufkont, kredit
"""

result2 = pd.read_sql(query2, conn)
print("Query 2 — Default rate by checking account status")
print(result2)

Query 2 — Default rate by checking account status
   checking_status  default_flag  total
0                1             0    135
1                1             1    139
2                2             0    105
3                2             1    164
4                3             0     14
5                3             1     49
6                4             0     46
7                4             1    348


## Query 3 — Average loan amount and duration by default status

**Business question:** Do longer loans carry higher default risk?

**Why it matters:** Loan duration is one of the most intuitive risk drivers 
in retail credit. This query uses CASE WHEN to create duration segments 
and computes the default rate per segment.

In [81]:
# Query 3 — Average loan amount and duration by default status
query3 = """
SELECT 
    kredit as default_flag,
    COUNT(*) as total,
    ROUND(AVG(hoehe), 2) as avg_amount,
    ROUND(AVG(laufzeit), 2) as avg_duration
FROM german_credit
GROUP BY kredit
"""

result3 = pd.read_sql(query3, conn)
print("Query 3 — Average loan amount and duration by default status")
print(result3)

Query 3 — Average loan amount and duration by default status
   default_flag  total  avg_amount  avg_duration
0             0    300     3938.13         24.86
1             1    700     2985.44         19.21


## Query 4 — High Risk Profile Identification

**Business question:** Which applicants combine multiple high-risk 
characteristics simultaneously?

**Why it matters:** In production, risk teams define rule-based filters 
to flag applicants for manual review before the scorecard is even applied. 
This query identifies applicants with no or negative checking account balance, 
long loan duration, and poor credit history.

In [82]:
# Query 4 — High risk profile identification
query4 = """
SELECT 
    laufkont,
    laufzeit,
    hoehe,
    moral,
    kredit
FROM german_credit
WHERE laufkont = 1
AND kredit = 0
ORDER BY hoehe DESC
LIMIT 20
"""

result4 = pd.read_sql(query4, conn)
print("Query 4 — High risk applicants (top 20)")
print(result4)

Query 4 — High risk applicants (top 20)
    laufkont  laufzeit  hoehe  moral  kredit
0          1         6  14896      2       0
1          1        30  11998      2       0
2          1        45  11816      0       0
3          1        48  10297      2       0
4          1        36   9629      4       0
5          1        36   9271      2       0
6          1        14   8978      2       0
7          1        36   8335      2       0
8          1        36   8229      2       0
9          1        36   8065      4       0
10         1        12   7865      2       0
11         1        48   7763      2       0
12         1        48   7685      1       0
13         1        18   7511      2       0
14         1        60   7297      2       0
15         1        42   7174      2       0
16         1        48   7119      0       0
17         1        48   6999      2       0
18         1        36   6887      3       0
19         1        24   6872      1       0


## Pipeline Connection

The SQL extraction layer validates data quality and confirms the dataset 
is ready for analysis. The clean dataframe is exported to 
`data/raw/german_credit_data.csv` and used directly on `01-eda.ipynb`.



In [83]:
# Pipeline connection — export clean dataframe for notebook 01
df_clean = pd.read_sql("SELECT * FROM german_credit", conn)
df_clean.to_csv('../data/raw/german_credit_data.csv', index=False)

print("Data extracted from SQL and ready for pipeline")
print(f"Shape: {df_clean.shape}")

conn.close()

Data extracted from SQL and ready for pipeline
Shape: (1000, 21)
